In [1]:
# import dataset from disk
from datasets import load_from_disk

ds = load_from_disk('cord-v2')

In [2]:
train, test, validation = ds['train'], ds['test'], ds['validation']

In [3]:
train

Dataset({
    features: ['image', 'ground_truth'],
    num_rows: 800
})


Extract relevant information for training from the ground truth.

The ground truth for a image is a list of dictionaries with the following keys:
'gt_parse', 'meta', 'valid_line', 'roi', 'repeating_symbol', 'dontcare'.

'valid_line' contains annotations that we care about for training.

For training, for each image, we need to extract the following information:
- image
- words
- boxes
- labels

Then, it is sent to the processor for encoding, 
i.e. converting words to tokens and labels to integers.


In [4]:
from tqdm.notebook import tqdm
import json

def normalize_bbox(bbox, width, height):
    return [
        int(1000 * (bbox[0] / width)),
        int(1000 * (bbox[1] / height)),
        int(1000 * (bbox[2] / width)),
        int(1000 * (bbox[3] / height)),
    ]

def build_input_for_processor(dataset):
    '''
    dataset is of type datasets.DatasetDict: Dataset({ features: ['image', 'ground_truth'] })
    '''
    images, words, boxes, labels = [], [], [], []

    for item in tqdm(dataset):
        words_item = []
        boxes_item = []
        labels_item = []
        image_item = item['image']
        
        ground_truth = json.loads(item['ground_truth'])

        width, height = ground_truth['meta']['image_size']['width'], ground_truth['meta']['image_size']['height']

        for annotations in ground_truth['valid_line']:
            label = annotations['category']

            for word in annotations['words']:
                text = word['text']
                # Skip empty text
                if text == '': continue

                x1 = word['quad']['x1']
                y1 = word['quad']['y1']
                x3 = word['quad']['x3']
                y3 = word['quad']['y3']

                box = [x1, y1, x3, y3]
                box = normalize_bbox(box, width=width, height=height)

                if min(box) < 0 or max(box) > 1000:
                    continue
                if ((box[3] - box[1]) < 0) or ((box[2] - box[0]) < 0):
                    continue
                    
                words_item.append(text)
                boxes_item.append(box)
                labels_item.append(label)
        
        images.append(image_item)
        words.append(words_item)
        boxes.append(boxes_item)
        labels.append(labels_item)


    return images, words, boxes, labels

In [5]:
# build input for train, test, validation
train_input = build_input_for_processor(train)

  0%|          | 0/800 [00:00<?, ?it/s]

In [6]:
test_input = build_input_for_processor(test)

  0%|          | 0/100 [00:00<?, ?it/s]

In [7]:
validation_input = build_input_for_processor(validation)

  0%|          | 0/100 [00:00<?, ?it/s]

In [8]:
images, words, boxes, labels = train_input
for w in zip(words[0], labels[0]):
    print(w)

('1', 'menu.cnt')
('x', 'menu.cnt')
('Nasi', 'menu.nm')
('Campur', 'menu.nm')
('Bali', 'menu.nm')
('75,000', 'menu.price')
('1', 'menu.cnt')
('x', 'menu.cnt')
('Bbk', 'menu.nm')
('Bengil', 'menu.nm')
('Nasi', 'menu.nm')
('125,000', 'menu.price')
('1', 'menu.cnt')
('x', 'menu.cnt')
('MilkShake', 'menu.nm')
('Starwb', 'menu.nm')
('37,000', 'menu.price')
('1', 'menu.cnt')
('x', 'menu.cnt')
('Ice', 'menu.nm')
('Lemon', 'menu.nm')
('Tea', 'menu.nm')
('24,000', 'menu.price')
('1', 'menu.cnt')
('x', 'menu.cnt')
('Nasi', 'menu.nm')
('Ayam', 'menu.nm')
('Dewata', 'menu.nm')
('70,000', 'menu.price')
('3', 'menu.cnt')
('x', 'menu.cnt')
('Free', 'menu.nm')
('Ice', 'menu.nm')
('Tea', 'menu.nm')
('0', 'menu.price')
('1', 'menu.cnt')
('x', 'menu.cnt')
('Organic', 'menu.nm')
('Green', 'menu.nm')
('Sa', 'menu.nm')
('65,000', 'menu.price')
('1', 'menu.cnt')
('x', 'menu.cnt')
('Ice', 'menu.nm')
('Tea', 'menu.nm')
('18,000', 'menu.price')
('1', 'menu.cnt')
('x', 'menu.cnt')
('Ice', 'menu.nm')
('Orange', '

In [28]:
type(images[0])

PIL.PngImagePlugin.PngImageFile

In [9]:
# Get all unique labels and count for each label across train, test, validation
from collections import Counter
import numpy as np

labels_train = np.concatenate(train_input[3])
labels_test = np.concatenate(test_input[3])
labels_validation = np.concatenate(validation_input[3])


all_labels = np.concatenate([labels_train, labels_test, labels_validation])
label_counts = Counter(all_labels)
label_counts

Counter({'menu.nm': 6594,
         'menu.price': 2589,
         'menu.cnt': 2423,
         'total.total_price': 2115,
         'sub_total.subtotal_price': 1480,
         'total.cashprice': 1393,
         'total.changeprice': 1297,
         'sub_total.tax_price': 1283,
         'menu.sub.nm': 831,
         'menu.unitprice': 750,
         'total.menuqty_cnt': 630,
         'total.creditcardprice': 410,
         'menu.discountprice': 403,
         'sub_total.service_price': 353,
         'sub_total.etc': 283,
         'menu.sub.cnt': 195,
         'sub_total.discount_price': 191,
         'menu.sub.price': 163,
         'total.emoneyprice': 131,
         'total.menutype_cnt': 130,
         'menu.num': 109,
         'total.total_etc': 87,
         'menu.etc': 19,
         'menu.sub.unitprice': 14,
         'menu.vatyn': 9,
         'menu.itemsubtotal': 7,
         'sub_total.othersvc_price': 6,
         'void_menu.nm': 3,
         'void_menu.price': 1})

In [10]:
max_label = max(label_counts, key=label_counts.get)
max_label

'menu.nm'

In [11]:
# convert label tuples to list
train_input = list(train_input)
test_input = list(test_input)
validation_input = list(validation_input)

In [ ]:
# Replace labels that are not total. and count is less than 200

def replace_labels(label):
    if label_counts[label] < 200 and not 'train' in label:
        return "O"
    return label

train_input[3] = [[replace_labels(label) for label in labels] for labels in train_input[3]]
test_input[3] = [[replace_labels(label) for label in labels] for labels in test_input[3]]
validation_input[3] = [[replace_labels(label) for label in labels] for labels in validation_input[3]]


In [14]:
labels_train = np.concatenate(train_input[3])
labels_test = np.concatenate(test_input[3])
labels_validation = np.concatenate(validation_input[3])


all_labels = np.concatenate([labels_train, labels_test, labels_validation])
label_counts = Counter(all_labels)
label_counts

Counter({'menu.nm': 6594,
         'menu.price': 2589,
         'menu.cnt': 2423,
         'total.total_price': 2115,
         'sub_total.subtotal_price': 1480,
         'total.cashprice': 1393,
         'total.changeprice': 1297,
         'sub_total.tax_price': 1283,
         'O': 1065,
         'menu.sub.nm': 831,
         'menu.unitprice': 750,
         'total.menuqty_cnt': 630,
         'total.creditcardprice': 410,
         'menu.discountprice': 403,
         'sub_total.service_price': 353,
         'sub_total.etc': 283})

In [18]:
unique_labels = list(set(all_labels))

In [19]:
unique_labels

['menu.cnt',
 'sub_total.etc',
 'total.menuqty_cnt',
 'menu.nm',
 'total.total_price',
 'menu.sub.nm',
 'total.changeprice',
 'sub_total.tax_price',
 'total.cashprice',
 'total.creditcardprice',
 'sub_total.service_price',
 'O',
 'menu.price',
 'sub_total.subtotal_price',
 'menu.unitprice',
 'menu.discountprice']

In [20]:
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for idx, label in enumerate(unique_labels)}
print(label2id)
print(id2label)

{'menu.cnt': 0, 'sub_total.etc': 1, 'total.menuqty_cnt': 2, 'menu.nm': 3, 'total.total_price': 4, 'menu.sub.nm': 5, 'total.changeprice': 6, 'sub_total.tax_price': 7, 'total.cashprice': 8, 'total.creditcardprice': 9, 'sub_total.service_price': 10, 'O': 11, 'menu.price': 12, 'sub_total.subtotal_price': 13, 'menu.unitprice': 14, 'menu.discountprice': 15}
{0: 'menu.cnt', 1: 'sub_total.etc', 2: 'total.menuqty_cnt', 3: 'menu.nm', 4: 'total.total_price', 5: 'menu.sub.nm', 6: 'total.changeprice', 7: 'sub_total.tax_price', 8: 'total.cashprice', 9: 'total.creditcardprice', 10: 'sub_total.service_price', 11: 'O', 12: 'menu.price', 13: 'sub_total.subtotal_price', 14: 'menu.unitprice', 15: 'menu.discountprice'}


In [29]:
from torch.utils.data import Dataset

class CORDV2Dataset(Dataset):
    """CORD V2 dataset."""

    def __init__(self, annotations, processor=None, max_length=256):
        """
        Args:
            annotations (List[List]): List of lists containing the word-level annotations (words, labels, boxes).
            processor (LayoutLMv3Processor): Processor to prepare the text + image.
        """
        self.images, self.words, self.boxes, self.labels = annotations
        self.processor = processor

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # first, take an image
        image = self.images[idx] # PIL.PngImagePlugin.PngImageFile

        # get word-level annotations 
        words = self.words[idx]
        boxes = self.boxes[idx]
        word_labels = self.labels[idx]

        assert len(words) == len(boxes) == len(word_labels)
        
        word_labels = [label2id[label] for label in word_labels]
        # use processor to prepare everything
        encoded_inputs = self.processor(image, words, boxes=boxes, word_labels=word_labels, 
                                        padding="max_length", truncation=True, 
                                        return_tensors="pt")
        
        # remove batch dimension
        for k,v in encoded_inputs.items():
          encoded_inputs[k] = v.squeeze()

        # assert encoded_inputs.input_ids.shape == torch.Size([512])
        # assert encoded_inputs.attention_mask.shape == torch.Size([512])
        # assert encoded_inputs.token_type_ids.shape == torch.Size([512])
        # assert encoded_inputs.bbox.shape == torch.Size([512, 4])
        # assert encoded_inputs.image.shape == torch.Size([3, 224, 224])
        # assert encoded_inputs.labels.shape == torch.Size([512]) 
      
        return encoded_inputs

In [22]:
from transformers import LayoutLMv3Processor, LayoutLMv3TokenizerFast, LayoutLMv3FeatureExtractor, LayoutLMv3ForTokenClassification

tokenizer = LayoutLMv3TokenizerFast.from_pretrained('microsoft/layoutlmv3-base')
processor = LayoutLMv3Processor(LayoutLMv3FeatureExtractor(apply_ocr=False), tokenizer)

Downloading:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/878k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/446k [00:00<?, ?B/s]

In [32]:
train_dataset = CORDV2Dataset(annotations=train_input,
                            processor=processor)
val_dataset = CORDV2Dataset(annotations=validation_input,
                            processor=processor)
test_dataset = CORDV2Dataset(annotations=test_input,
                         processor=processor)

In [33]:
encoding = train_dataset[0]

In [34]:
for k,v in encoding.items():
  print(k, v.shape)

input_ids torch.Size([512])
attention_mask torch.Size([512])
bbox torch.Size([512, 4])
labels torch.Size([512])
pixel_values torch.Size([3, 224, 224])


In [35]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=2, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=2)

In [37]:
# store the dataloaders locally
import pickle

with open('train_dataloader.pkl', 'wb') as f:
    pickle.dump(train_dataloader, f)

with open('val_dataloader.pkl', 'wb') as f:
    pickle.dump(val_dataloader, f)

with open('test_dataloader.pkl', 'wb') as f:
    pickle.dump(test_dataloader, f)

In [39]:
model = LayoutLMv3ForTokenClassification.from_pretrained('microsoft/layoutlmv3-base', num_labels=len(unique_labels))

Downloading:   0%|          | 0.00/856 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/478M [00:00<?, ?B/s]

Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [40]:
from transformers import AdamW
import torch
from tqdm.notebook import tqdm as tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

global_step = 0
num_train_epochs = 4

#put the model in training mode
model.train() 
for epoch in range(num_train_epochs):  
   print("Epoch:", epoch)
   for batch in tqdm(train_dataloader):
        # get the inputs;
        input_ids = batch['input_ids'].to(device)
        bbox = batch['bbox'].to(device)
        pixel_values = batch['pixel_values'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # zero the parameter gradients
        optimizer.zero_grad()
        
        # forward + backward + optimize
        outputs = model(input_ids=input_ids,
                        bbox=bbox,
                        pixel_values=pixel_values,
                        attention_mask=attention_mask,
                        labels=labels) 
        loss = outputs.loss
        
        # print loss every 100 steps
        if global_step % 100 == 0:
          print(f"Loss after {global_step} steps: {loss.item()}")

        loss.backward()
        optimizer.step()
        global_step += 1

#save pre trained locally
model.save_pretrained('cord-v2-layoutlmv3-base')


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch: 0


  0%|          | 0/400 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:699: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


Loss after 0 steps: 2.7856249809265137
Loss after 100 steps: 0.24086493253707886
Loss after 200 steps: 0.6747856140136719
Loss after 300 steps: 0.05047694221138954
Epoch: 1


...

Loss after 400 steps: 0.0895557701587677
Loss after 500 steps: 0.22926723957061768
Loss after 600 steps: 0.12444775551557541
Loss after 700 steps: 0.07612026482820511
Epoch: 2


  0%|          | 0/400 [00:00<?, ?it/s]

Loss after 800 steps: 0.2952940762042999
Loss after 900 steps: 0.4259636104106903
Loss after 1000 steps: 0.18437401950359344
Loss after 1100 steps: 0.22080090641975403
Epoch: 3


  0%|          | 0/400 [00:00<?, ?it/s]

Loss after 1200 steps: 0.016868269070982933
Loss after 1300 steps: 0.1394503116607666
Loss after 1400 steps: 0.22702789306640625
Loss after 1500 steps: 0.023238586261868477


In [47]:
test_sample = test_input[0]

test_sample

[<PIL.PngImagePlugin.PngImageFile image mode=RGB size=432x648>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=960x1280>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=864x1296>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=864x1296>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=864x1296>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=2304x4096>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=864x1296>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=576x864>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=864x1296>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=576x864>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=576x864>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=576x864>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=576x864>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=864x1296>,
 <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1836x3264>,
 <PIL.PngImagePlugin.PngImage

In [42]:
encoding = test_dataset[0]
processor.tokenizer.decode(encoding['input_ids'])

for k,v in encoding.items():
  encoding[k] = v.unsqueeze(0).to(device)

model.eval()
# forward pass
outputs = model(input_ids=encoding['input_ids'],
                attention_mask=encoding['attention_mask'],
                bbox=encoding['bbox'],
                pixel_values=encoding['pixel_values'])

/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:699: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


In [43]:
prediction_indices = outputs.logits.argmax(-1).squeeze().tolist()
print(prediction_indices)

[6, 11, 11, 11, 3, 3, 3, 3, 3, 0, 14, 14, 14, 12, 12, 12, 11, 11, 11, 11, 11, 13, 11, 11, 7, 7, 7, 7, 7, 13, 13, 13, 13, 13, 4, 4, 4, 4, 2, 2, 2, 9, 9, 11, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 6, 6, 8, 6, 6, 6, 6, 6, 6, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8

In [44]:
prediction_indices = outputs.logits.argmax(-1).squeeze().tolist()
predictions = [id2label[label] for gt, label in zip(encoding['labels'].squeeze().tolist(), prediction_indices) if gt != -100]
print(predictions)

['O', 'menu.nm', 'menu.nm', 'menu.cnt', 'menu.unitprice', 'menu.price', 'O', 'O', 'O', 'O', 'sub_total.tax_price', 'sub_total.tax_price', 'sub_total.subtotal_price', 'sub_total.subtotal_price', 'total.total_price', 'total.total_price', 'total.menuqty_cnt', 'total.creditcardprice', 'total.creditcardprice', 'total.creditcardprice', 'total.creditcardprice', 'total.creditcardprice', 'total.creditcardprice', 'total.creditcardprice']


In [48]:
test_input[3][0]

['O',
 'menu.nm',
 'menu.nm',
 'menu.cnt',
 'menu.price',
 'O',
 'O',
 'O',
 'O',
 'O',
 'sub_total.tax_price',
 'sub_total.tax_price',
 'sub_total.subtotal_price',
 'sub_total.subtotal_price',
 'total.total_price',
 'total.total_price',
 'total.menuqty_cnt',
 'total.menuqty_cnt',
 'total.creditcardprice',
 'total.creditcardprice',
 'total.creditcardprice',
 'total.creditcardprice',
 'total.creditcardprice',
 'total.creditcardprice']